# Blending berbobot (OOF)
**tabular › ensembling_regression** · notebook `oof_blend`

**Kapan dipakai:** Menggabungkan beberapa model dengan bobot yang dicari dari prediksi out-of-fold. Cara ensembling paling umum di lomba.

**Jebakan umum:** Bobot HARUS dicari dari OOF training, bukan dari validation/test. Ensembel dari model yang mirip tidak menambah apa-apa; cek korelasi prediksi.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular/regression/diamonds_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular/regression/diamonds_test.csv"      # None kalau tidak ada file test
TARGET     = "price"
ID_COL     = 'id'
DROP_COLS  = []
GROUP_COL  = None                      # isi bila ada grup agar tidak bocor
SCORING    = "neg_root_mean_squared_error"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "oof_blend"
TASK       = "regression"
KIND       = "tree"
OUT = out_dir("tabular", "ensembling_regression", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
y = df[TARGET].to_numpy(dtype=float)
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Model dasar yang beragam
Ensembel hanya berguna bila kesalahannya berbeda; makin beragam makin baik (pohon, linear, boosting).

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
BASE = {
    "lgbm":   (LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbose=-1), "tree"),
    "rf":     (RandomForestRegressor(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=SEED), "tree"),
    "extra":  (ExtraTreesRegressor(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=SEED), "tree"),
    "linear": (Ridge(alpha=1.0), "linear"),
}
pipes = {k: build_pipeline(mdl, NUM, CAT, kind=kind) for k, (mdl, kind) in BASE.items()}

## 5. Prediksi out-of-fold (OOF) pada training + prediksi validation
**OOF**: tiap baris training diprediksi oleh model yang tidak pernah melihat baris itu. Bobot/meta-model dicari dari OOF, jadi validation tetap bersih.

In [ ]:
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_predict
cv = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED) if TASK == "classification" else KFold(N_SPLITS, shuffle=True, random_state=SEED)
def pred(pp, X_):
    return pp.predict_proba(X_)[:, 1] if TASK == "classification" else pp.predict(X_)
OOF = pd.DataFrame({k: (cross_val_predict(pp, Xtr, ytr, cv=cv, method="predict_proba")[:, 1] if TASK == "classification"
                       else cross_val_predict(pp, Xtr, ytr, cv=cv)) for k, pp in pipes.items()}, index=Xtr.index)
for pp in pipes.values():
    pp.fit(Xtr, ytr)
VAL = pd.DataFrame({k: pred(pp, Xva) for k, pp in pipes.items()}, index=Xva.index)
print("korelasi prediksi OOF antar model (makin rendah makin berguna untuk ensemble):")
display(OOF.corr().round(3))

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
def score(yt, pr):
    return {"rmse": float(np.sqrt(mean_squared_error(yt, pr))), "mae": mean_absolute_error(yt, pr)}

## 6. Gabungkan

In [ ]:
from superdb.tabular_extra_b import blend_weights
w = blend_weights(OOF.values, ytr, TASK)
print("bobot (dari OOF):", dict(zip(OOF.columns, w.round(3))))
def combine(P):
    return np.asarray(P)[:, :len(w)] @ w

## 7. Apakah ensemble benar-benar menang pada validation?

In [ ]:
rows = {k: score(yva, VAL[k].to_numpy()) for k in VAL}
rows["rata-rata sederhana"] = score(yva, VAL.mean(axis=1).to_numpy())
rows[MODEL_KEY.upper()] = score(yva, combine(VAL.values))
res = pd.DataFrame(rows).T
display(res.sort_values(res.columns[0], ascending=(TASK == "regression")).round(4))

## 8. Latih ulang model dasar pada semua data → submission

In [ ]:
from sklearn.base import clone
full = {k: clone(pp).fit(X, y) for k, pp in pipes.items()}
if test is not None:
    Xte, _, _ = prepare_features(test, cols); Xte = Xte.reindex(columns=X.columns)
    PT = np.column_stack([pred(full[k], Xte) for k in OOF.columns])
    fp = combine(PT)
    ids = test[ID_COL] if ID_COL else np.arange(len(test))
    sub = make_submission(ids, fp, ID_COL or "id", TARGET, OUT / "submission.csv"); display(sub.head(3))
save_json({"method": MODEL_KEY, "valid": res.to_dict("index")}, OUT / "metrics.json")
save_model({"pipes": full, "cols": cols}, OUT / "model.joblib"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Kalau ensemble hanya naik sangat tipis, model dasarnya terlalu mirip: tambahkan model dari keluarga berbeda (mis. CatBoost, KNN, MLP).
- Untuk lomba: simpan OOF dan prediksi test tiap model ke disk agar bisa diblend ulang tanpa melatih lagi.
- Klasifikasi di notebook ini biner; untuk multikelas, blend vektor probabilitas per kelas.